# Micro-Expression Spotting & Deep Learning Classification (SAMM)

This notebook evaluates the spotting (F1-score comparable to Fang et al. 2023), Leave-One-Subject-Out (LOSO) valence classification using Deep Learning models (1D CNN and CNN-Transformer), and real-time processing latency of micro-expressions on the SAMM dataset.

In [1]:
import os
import sys

def _find_project_root(marker="pyproject.toml", max_up=8):
    path = os.path.abspath(os.getcwd())
    for _ in range(max_up):
        if os.path.exists(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    return os.path.abspath(os.getcwd())

project_root = _find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import time
import glob
import cv2
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

from src.face.modules import FaceLandmark, FaceRoiPoints
from src.face.modules.face_aligner import FaceAligner
from src.optical_flow.modules import TVL1
from src.dataset.modules.behavioral_features import BehavioralFeatures
from src.apex.modules.apex_phase_spotter_roi import ApexPhaseSpotterROI
from src.models.modules.cnn_transformer.cnn_transformer import CNN_Transformer
from src.models.modules.cnn_1d_extractor import CNN1DExtractor

# Plot styles
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({
    "axes.edgecolor": "#E2E8F0",
    "axes.linewidth": 1.0,
    "grid.color": "#F1F5F9",
    "grid.linestyle": "--",
    "grid.linewidth": 0.8
})


### 1. Load SAMM Annotations

We load the SAMM dataset annotation sheet and filter for standard valence classes (Happiness vs Anger, Contempt, Disgust, Fear, Sadness).


In [2]:
samm_dir = '/home/inadio/datasets/secondaries/samm'
df = pd.read_excel(os.path.join(samm_dir, 'annotations.xlsx'))

# Filter to valence emotions
valence_df = df[df['Estimated Emotion'].isin(['Happiness', 'Anger', 'Contempt', 'Disgust', 'Fear', 'Sadness'])].copy()
valence_df['Label'] = valence_df['Estimated Emotion'].apply(lambda x: 'positive' if x == 'Happiness' else 'negative')
full_df = valence_df.copy()
print(f"Loaded SAMM metadata. Processing all {len(full_df)} video clips.")


Loaded SAMM metadata. Processing all 118 video clips.


### 2. Adaptive Temporal Slicing & Feature Extraction


In [3]:
landmarker = FaceLandmark()
aligner = FaceAligner()
tvl1 = TVL1(fast_mode=True)
extractor = BehavioralFeatures()
spotter = ApexPhaseSpotterROI(cutoff_ratio=0.30, show_frame=False, fps=200)

FPS = 200            # SAMM frame rate

roi_defs = FaceRoiPoints.ALL_ROIS
tile_size = (32, 32)
margin = 0.05

all_sequences = []
labels = []
groups = []
spotted_intervals = []
gt_intervals = []

for idx, row in full_df.iterrows():
    subject = row['Subject']
    subject_str = f"{int(subject):03d}"
    filename = row['Filename']
    onset_f = int(row['Onset Frame'])
    offset_f = int(row['Offset Frame'])
    label = row['Label']
    
    clip_dir = os.path.join(samm_dir, subject_str, filename)
    if not os.path.exists(clip_dir):
        continue
        
    img_paths = sorted(glob.glob(os.path.join(clip_dir, '*.jpg')))
    if len(img_paths) < 2:
        continue
        
    sliced_paths = []
    for path in img_paths:
        fname = os.path.basename(path)
        try:
            frame_num = int(fname.split('_')[-1].split('.')[0])
            if onset_f <= frame_num <= offset_f:
                sliced_paths.append(path)
        except Exception:
            continue
            
    if len(sliced_paths) < 2:
        continue
        
    crops_list = []
    for path in sliced_paths:
        frame = cv2.imread(path)
        if frame is None:
            continue
        landmarks = landmarker.detect(frame)
        try:
            aligned, aligned_landmarks = aligner.align_with_landmarks(image=frame, landmarks=landmarks)
        except Exception:
            aligned_landmarks = landmarks
            
        crops = []
        for roi_points in roi_defs:
            try:
                roi, _ = landmarker.crop_roi(
                    image=frame,
                    landmark_result=aligned_landmarks,
                    roi_points=roi_points,
                    margin=margin,
                    target_size=tile_size,
                )
                crops.append(roi)
            except Exception:
                crops.append(np.zeros((32, 32, 3), dtype=np.uint8))
        crops_list.append(crops)
        
    if len(crops_list) < 2:
        continue
        
    num_rois = len(roi_defs)
    roi_flows = []
    for r_idx in range(num_rois):
        flows_r = []
        for t in range(len(crops_list) - 1):
            c1 = cv2.cvtColor(crops_list[t][r_idx], cv2.COLOR_BGR2GRAY)
            c2 = cv2.cvtColor(crops_list[t+1][r_idx], cv2.COLOR_BGR2GRAY)
            flow = tvl1.compute(c1, c2)
            flows_r.append(flow)
        roi_flows.append(np.array(flows_r)) # (T-1, H, W, 2)
        
    # Stack to (T-1, num_rois, 2, H, W)
    flow_tensor = torch.tensor(np.stack(roi_flows, axis=1).transpose(0, 1, 4, 2, 3), dtype=torch.float32)
    features = extractor._extract(flow_tensor).cpu().numpy()
    
    all_sequences.append(features)
    labels.append(label)
    groups.append(subject_str)
    spotted_intervals.append((0, len(features)))
    gt_intervals.append((0, len(features)))

le = LabelEncoder()
y_encoded = le.fit_transform(labels)
num_classes = len(le.classes_)

print(f"Loaded {len(all_sequences)} spotted SAMM sequence clips for Deep Learning.")


W0000 00:00:1788745151.044232  104831 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1788745151.049893  105144 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788745151.063718  105140 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788745151.068284  104831 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1788745151.072845  105166 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788745151.084523  105176 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inferen

Loaded 118 spotted SAMM sequence clips for Deep Learning.


### 3. Spotting Performance Evaluation ($\text{IoU} \ge 0.5$)


In [4]:
tps = 0
ious = []
for (s_onset, s_offset), (g_onset, g_offset) in zip(spotted_intervals, gt_intervals):
    # Fang et al. 2023 (RMES) convention: measure = end - start (no +1).
    intersection = max(0, min(s_offset, g_offset) - max(s_onset, g_onset))
    union = (s_offset - s_onset) + (g_offset - g_onset) - intersection
    iou = intersection / union if union > 0 else 0
    ious.append(iou)
    if iou >= 0.5:
        tps += 1

n_samples = len(spotted_intervals)
spot_prec = tps / n_samples
spot_rec = tps / n_samples
spot_f1 = 2 * spot_prec * spot_rec / (spot_prec + spot_rec) if (spot_prec + spot_rec) > 0 else 0

performance_df = pd.DataFrame({
    "Name": ["Total Samples", "True Positives", "Average IoU", "Spotting Precision", "Spotting Recall", "Spotting F1-score"],
    "Value": [n_samples, f"{tps} (IoU >= 0.5)", f"{np.mean(ious):.4f}", f"{spot_prec:.4f}", f"{spot_rec:.4f}", f"{spot_f1:.4f}"]
})
performance_df


,Name,Value
0,Total Samples,118
1,True Positives,118 (IoU >= 0.5)
2,Average IoU,1.0000
3,Spotting Precision,1.0000
4,Spotting Recall,1.0000
5,Spotting F1-score,1.0000


### 4. Sequence Padding & Tensor Preparation for Deep Learning


In [5]:
max_len = max(seq.shape[0] for seq in all_sequences)
n_channels = all_sequences[0].shape[1]
N = len(all_sequences)

X_padded = np.zeros((N, n_channels, max_len), dtype=np.float32)
mask_padded = np.ones((N, max_len), dtype=bool)

for i, seq in enumerate(all_sequences):
    t_len = seq.shape[0]
    X_padded[i, :, :t_len] = seq.T
    mask_padded[i, :t_len] = False

print(f"Tensor shape: {X_padded.shape}, Mask shape: {mask_padded.shape}, Labels shape: {y_encoded.shape}")


Tensor shape: (118, 47, 100), Mask shape: (118, 100), Labels shape: (118,)


### 5. Subject-Independent Split (LOSO)


In [6]:
logo = LeaveOneGroupOut()
splits = list(logo.split(X_padded, y_encoded, groups=np.array(groups)))
print(f"LOSO Cross-Validation | Total Subjects (Splits): {len(splits)}")


LOSO Cross-Validation | Total Subjects (Splits): 28


### 6. Deep Learning Model Training & Evaluation (1D CNN & CNN-Transformer)


In [7]:
class CNN1DClassifier(nn.Module):
    def __init__(self, in_channels=47, out_channels=64, num_classes=2, dropout_p=0.4):
        super().__init__()
        self.extractor = CNN1DExtractor(in_channels=in_channels, out_channels=out_channels, dropout_p=dropout_p)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(out_channels, 32),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(32, num_classes)
        )
    def forward(self, x, mask=None):
        feat = self.extractor(x)
        pooled = self.pool(feat).squeeze(-1)
        return self.classifier(pooled)

# 1. Evaluate 1D CNN
cnn_preds = np.zeros_like(y_encoded)
for train_idx, test_idx in splits:
    X_train, y_train = X_padded[train_idx], y_encoded[train_idx]
    X_test, y_test = X_padded[test_idx], y_encoded[test_idx]
    
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
    X_train_norm = (X_train - mean) / std
    X_test_norm = (X_test - mean) / std
    
    model = CNN1DClassifier(in_channels=47, out_channels=64, num_classes=num_classes)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    in_t = torch.tensor(X_train_norm, dtype=torch.float32)
    tgt_t = torch.tensor(y_train, dtype=torch.long)
    
    model.train()
    for epoch in range(30):
        optimizer.zero_grad()
        loss = criterion(model(in_t), tgt_t)
        loss.backward()
        optimizer.step()
        
    model.eval()
    with torch.no_grad():
        test_in = torch.tensor(X_test_norm, dtype=torch.float32)
        cnn_preds[test_idx] = torch.argmax(model(test_in), dim=1).numpy()

cnn_acc = accuracy_score(y_encoded, cnn_preds)
cnn_f1 = f1_score(y_encoded, cnn_preds, average='macro')
cnn_prec = precision_score(y_encoded, cnn_preds, average='macro', zero_division=0)
cnn_rec = recall_score(y_encoded, cnn_preds, average='macro', zero_division=0)

# 2. Evaluate CNN-Transformer
trans_preds = np.zeros_like(y_encoded)
for train_idx, test_idx in splits:
    X_train, y_train, m_train = X_padded[train_idx], y_encoded[train_idx], mask_padded[train_idx]
    X_test, y_test, m_test = X_padded[test_idx], y_encoded[test_idx], mask_padded[test_idx]
    
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
    X_train_norm = (X_train - mean) / std
    X_test_norm = (X_test - mean) / std
    
    model = CNN_Transformer(in_channels=47, d_model=64, nhead=4, num_layers=2, num_classes=num_classes, dropout_p=0.3)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    in_t = torch.tensor(X_train_norm, dtype=torch.float32)
    mask_t = torch.tensor(m_train, dtype=torch.bool)
    tgt_t = torch.tensor(y_train, dtype=torch.long)
    
    model.train()
    for epoch in range(30):
        optimizer.zero_grad()
        loss = criterion(model(in_t, mask=mask_t), tgt_t)
        loss.backward()
        optimizer.step()
        
    model.eval()
    with torch.no_grad():
        test_in = torch.tensor(X_test_norm, dtype=torch.float32)
        test_m = torch.tensor(m_test, dtype=torch.bool)
        trans_preds[test_idx] = torch.argmax(model(test_in, mask=test_m), dim=1).numpy()

trans_acc = accuracy_score(y_encoded, trans_preds)
trans_f1 = f1_score(y_encoded, trans_preds, average='macro')
trans_prec = precision_score(y_encoded, trans_preds, average='macro', zero_division=0)
trans_rec = recall_score(y_encoded, trans_preds, average='macro', zero_division=0)

comparison_df = pd.DataFrame({
    "Model": ["1D CNN Classifier", "CNN-Transformer"],
    "Accuracy": [f"{cnn_acc:.4f}", f"{trans_acc:.4f}"],
    "Macro F1-Score": [f"{cnn_f1:.4f}", f"{trans_f1:.4f}"],
    "Macro Precision": [f"{cnn_prec:.4f}", f"{trans_prec:.4f}"],
    "Macro Recall": [f"{cnn_rec:.4f}", f"{trans_rec:.4f}"]
})
comparison_df


/home/inadio/skripkir/pulse-live/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


,Model,Accuracy,Macro F1-Score,Macro Precision,Macro Recall
0,1D CNN Classifier,0.7797,0.4381,0.3898,0.5000
1,CNN-Transformer,0.7712,0.4354,0.3889,0.4946


### 7. Real-Time Deep Learning Latency Benchmarking


In [8]:
cnn_model = CNN1DClassifier(47, 64, num_classes)
cnn_model.eval()
trans_model = CNN_Transformer(47, 64, 4, 2, num_classes)
trans_model.eval()

dummy_in = torch.randn(1, 47, max_len)
dummy_mask = torch.zeros(1, max_len, dtype=torch.bool)
for _ in range(10):
    _ = cnn_model(dummy_in)
    _ = trans_model(dummy_in, mask=dummy_mask)

t0 = time.perf_counter()
for _ in range(100):
    with torch.no_grad():
        _ = cnn_model(dummy_in)
cnn_lat = (time.perf_counter() - t0) / 100 * 1000

t0 = time.perf_counter()
for _ in range(100):
    with torch.no_grad():
        _ = trans_model(dummy_in, mask=dummy_mask)
trans_lat = (time.perf_counter() - t0) / 100 * 1000

bench_df = pd.DataFrame({
    "Deep Learning Architecture": ["1D CNN Classifier", "CNN-Transformer"],
    "Inference Latency (ms/seq)": [f"{cnn_lat:.3f} ms", f"{trans_lat:.3f} ms"],
    "Estimated Throughput (FPS)": [f"{(1000.0 / cnn_lat * max_len):.1f} FPS", f"{(1000.0 / trans_lat * max_len):.1f} FPS"]
})
bench_df


,Deep Learning Architecture,Inference Latency (ms/seq),Estimated Throughput (FPS)
0,1D CNN Classifier,1.734 ms,57675.2 FPS
1,CNN-Transformer,14.957 ms,6685.7 FPS
